<a href="https://colab.research.google.com/github/MuhammadRafiq215/CNN_Basics_Fashion_MNIST.ipynb/blob/main/RNN_Workbook_How_Data_Goes_Into_an_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNN Workbook: How Data Is Given to a Recurrent Neural Network

**A step-by-step book for students**

You have already learned what an RNN is. The hard part in practice is not the
model. The hard part is getting your data into the exact shape the RNN wants.

This workbook answers one question, slowly and completely:

> **How does a sentence of text become something an RNN can read?**

### How to use this notebook

* Run the cells one by one, from top to bottom.
* After every cell, look at the printed **shape**. The shape is the story.
* Do not skip the small toy example. It uses only 8 sentences, so you can see
  every single number with your own eyes.
* The real dataset comes later, after you understand the small one.

### What is inside

| Part | What you learn |
| --- | --- |
| 1 | The one shape rule for every RNN |
| 2 | A tiny text dataset you can read by hand |
| 3 | Step 1 - clean the text |
| 4 | Step 2 - split into tokens |
| 5 | Step 3 - build the vocabulary |
| 6 | Step 4 - turn words into numbers |
| 7 | Step 5 - padding |
| 8 | Step 6 - the Embedding layer makes it 3D |
| 9 | The whole journey in one picture |
| 10 | Step 7 - feed it to a SimpleRNN |
| 11 | A real dataset: IMDB movie reviews |
| 12 | Build and train the RNN |
| 13 | Evaluate and predict your own sentence |
| 14 | SimpleRNN vs LSTM vs GRU |
| 15 | Common mistakes |
| 16 | Exercises |


---
## Setup

Run this cell first. It loads the libraries and prints their versions.
If you are on Google Colab, everything here is already installed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

print("NumPy      :", np.__version__)
print("Pandas     :", pd.__version__)
print("TensorFlow :", tf.__version__)
print("Keras      :", keras.__version__)

In [ ]:
# Fix the random seed so you get the same numbers every time you run this.
keras.utils.set_random_seed(42)

# Make the plots look clean.
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("Setup done.")

---
# Part 1. The one shape rule

An RNN always wants its input in **three dimensions**:

```
(batch_size, timesteps, features)
```

| Dimension | Meaning | Example with text |
| --- | --- | --- |
| `batch_size` | How many samples | 8 sentences |
| `timesteps` | How many steps in each sample | 6 words per sentence |
| `features` | How many numbers describe one step | 10 numbers per word |

Read it like this: **"8 sentences, each 6 words long, each word described by 10 numbers."**

The RNN reads one timestep at a time. At each timestep it takes one vector of
`features` numbers, updates its memory, and moves to the next timestep.

Everything we do in Parts 3 to 8 exists only to build this 3D box.

In [ ]:
# A fake example of the 3D shape, so you can see it before we build a real one.
fake_input = np.zeros((8, 6, 10))

print("Shape        :", fake_input.shape)
print("Samples      :", fake_input.shape[0])
print("Timesteps    :", fake_input.shape[1])
print("Features     :", fake_input.shape[2])

In [ ]:
# One sample = one sentence = a 2D table of (timesteps, features).
one_sample = fake_input[0]
print("One sentence shape:", one_sample.shape)

# One timestep = one word = a 1D vector of features.
one_word = fake_input[0][0]
print("One word shape    :", one_word.shape)

**Remember this:** text is 1D (a string), your prepared data is 2D
(sentences x words), and the RNN input is 3D (sentences x words x numbers).
The Embedding layer is what adds the third dimension. We will see that in Part 8.

---
# Part 2. A tiny dataset you can read by hand

We start with 8 short movie reviews. This is small on purpose. With 8 sentences
you can check every number yourself. Once you trust the process, we will use
25,000 real reviews.

* Label `1` means a positive review.
* Label `0` means a negative review.

In [ ]:
sentences = [
    "The movie was great!",
    "I loved the acting.",
    "What a wonderful film",
    "The story was boring.",
    "I hated the ending!",
    "A terrible waste of time",
    "The music was beautiful",
    "Worst film I have seen",
]

labels = [1, 1, 1, 0, 0, 0, 1, 0]

print("Number of sentences:", len(sentences))
print("Number of labels   :", len(labels))

In [ ]:
# Put it in a DataFrame so it is easy to read.
toy_df = pd.DataFrame({"review": sentences, "label": labels})
toy_df

In [ ]:
# Always check the balance of your labels.
print(toy_df["label"].value_counts())

---
# Part 3. Step 1 - Clean the text

Computers treat `"Great!"`, `"great"` and `"great."` as three different words.
That wastes vocabulary space and makes learning harder.

So we do two simple things:

1. Make everything lowercase.
2. Remove punctuation.

Keep cleaning simple at the start. You can always add more later.

In [ ]:
import string

print("Punctuation we will remove:")
print(string.punctuation)

In [ ]:
# Clean one sentence, so you can see exactly what happens.
example = "The movie was great!"

step_a = example.lower()
print("Original  :", example)
print("Lowercase :", step_a)

In [ ]:
# Now remove the punctuation characters.
step_b = ""
for ch in step_a:
    if ch not in string.punctuation:
        step_b = step_b + ch

print("No punctuation:", step_b)

In [ ]:
# Do the same for all 8 sentences.
clean_sentences = []

for s in sentences:
    s = s.lower()
    cleaned = ""
    for ch in s:
        if ch not in string.punctuation:
            cleaned = cleaned + ch
    clean_sentences.append(cleaned)

clean_sentences

In [ ]:
# Compare before and after, side by side.
pd.DataFrame({"before": sentences, "after": clean_sentences})

---
# Part 4. Step 2 - Split into tokens

A **token** is one unit of text. Here one token = one word.

Splitting on spaces is the simplest tokenizer, and it is enough to learn with.
Real systems use subword tokenizers, which you will meet when you study
transformers.

In [ ]:
# Split one sentence into words.
print(clean_sentences[0].split())

In [ ]:
# Split all of them.
tokenized = []

for s in clean_sentences:
    tokenized.append(s.split())

for t in tokenized:
    print(t)

In [ ]:
# How long is each sentence, in words?
lengths = []
for t in tokenized:
    lengths.append(len(t))

print("Lengths      :", lengths)
print("Longest      :", max(lengths))
print("Shortest     :", min(lengths))

Notice that the sentences have **different lengths**: some have 4 words,
some have 5. A neural network needs a rectangle, not a ragged list.
We fix this in Part 7 with padding.

---
# Part 5. Step 3 - Build the vocabulary

The vocabulary is a dictionary that maps every word to a number:

```
"the" -> 2, "movie" -> 3, "was" -> 4, ...
```

Two index numbers are reserved, and this is a very common convention:

| Index | Name | Why it exists |
| --- | --- | --- |
| `0` | `<PAD>` | The filler we add to short sentences |
| `1` | `<OOV>` | "Out Of Vocabulary" - any word the model never saw in training |

Real words start at index `2`.

We build this dictionary by hand here. Later, Keras can do it for you with the
`TextVectorization` layer, but you should see it done manually once so you know
there is no magic inside.

In [ ]:
from collections import Counter

# Put every word from every sentence into one long list.
all_words = []
for t in tokenized:
    for w in t:
        all_words.append(w)

print("Total words        :", len(all_words))
print("Different words    :", len(set(all_words)))

In [ ]:
# Count how often each word appears. Most common first.
word_counts = Counter(all_words)
word_counts.most_common()

In [ ]:
# Build the word -> number dictionary.
word_index = {}
word_index["<PAD>"] = 0
word_index["<OOV>"] = 1

next_number = 2
for word, count in word_counts.most_common():
    word_index[word] = next_number
    next_number = next_number + 1

print("Vocabulary size:", len(word_index))

In [ ]:
# Look at the dictionary.
for word in word_index:
    print(word_index[word], "->", word)

In [ ]:
# We also need the reverse dictionary, to go from numbers back to words.
index_word = {}
for word in word_index:
    index_word[word_index[word]] = word

print(index_word[2], index_word[3], index_word[4])

**Important:** the vocabulary must be built from the **training data only**.
If you build it using the test data too, your model has secretly seen the test
set. That is called data leakage, and your results will look better than they
really are.

---
# Part 6. Step 4 - Turn words into numbers

Now we replace every word with its number from the vocabulary.
This is called **encoding** or **vectorising** the text.

A word that is not in the dictionary becomes `1`, the `<OOV>` index.

In [ ]:
# Encode one sentence, slowly.
one_sentence = tokenized[0]
print("Words:", one_sentence)

one_sequence = []
for word in one_sentence:
    if word in word_index:
        one_sequence.append(word_index[word])
    else:
        one_sequence.append(word_index["<OOV>"])

print("Numbers:", one_sequence)

In [ ]:
# Encode all 8 sentences.
sequences = []

for t in tokenized:
    seq = []
    for word in t:
        if word in word_index:
            seq.append(word_index[word])
        else:
            seq.append(word_index["<OOV>"])
    sequences.append(seq)

for i in range(len(sequences)):
    print(tokenized[i], "->", sequences[i])

In [ ]:
# Check that we can go back from numbers to words.
back_to_words = []
for number in sequences[0]:
    back_to_words.append(index_word[number])

print("Numbers :", sequences[0])
print("Words   :", back_to_words)

In [ ]:
# What happens to a word the model has never seen?
new_sentence = "the cinematography was stunning"
new_tokens = new_sentence.split()

new_sequence = []
for word in new_tokens:
    if word in word_index:
        new_sequence.append(word_index[word])
    else:
        new_sequence.append(word_index["<OOV>"])

print("Words   :", new_tokens)
print("Numbers :", new_sequence)
print()
print("'cinematography' and 'stunning' were not in our 8 sentences,")
print("so they both became 1, which is <OOV>.")

**These numbers are only names, not values.** The word with index 12 is not
"bigger" or "better" than the word with index 3. They are just labels.
This is exactly why we need the Embedding layer in Part 8: it turns each label
into a vector of real numbers that the model can actually learn from.

---
# Part 7. Step 5 - Padding

Our sequences still have different lengths. A neural network needs every row to
be the same length, so we **pad** the short ones with zeros.

Two choices you must make:

| Choice | Options | Usual answer for RNN |
| --- | --- | --- |
| Where to add the zeros | `padding='pre'` or `'post'` | `'pre'` |
| Where to cut long texts | `truncating='pre'` or `'post'` | `'post'` |

Why `padding='pre'` for an RNN? The RNN reads left to right and its memory is
strongest at the end. If the zeros come last, the model finishes on a run of
nothing and partly forgets the real words. Putting the zeros first keeps the
real words closest to the end.

In [ ]:
# Look at the lengths again before padding.
for i in range(len(sequences)):
    print("length", len(sequences[i]), " ", sequences[i])

In [ ]:
# Plot the sentence lengths. On a real dataset, this plot decides your maxlen.
plt.hist(lengths, bins=range(min(lengths), max(lengths) + 2), edgecolor="black")
plt.xlabel("Words in a sentence")
plt.ylabel("How many sentences")
plt.title("Sentence lengths in the toy dataset")
plt.show()

In [ ]:
# Keras does the padding for us.
from keras.utils import pad_sequences

MAX_LEN = 6   # every sentence will become exactly 6 numbers long

padded_pre = pad_sequences(sequences, maxlen=MAX_LEN, padding="pre", truncating="post")

print("padding='pre'  (zeros at the start)")
print(padded_pre)

In [ ]:
# The same data with the zeros at the end, so you can see the difference.
padded_post = pad_sequences(sequences, maxlen=MAX_LEN, padding="post", truncating="post")

print("padding='post'  (zeros at the end)")
print(padded_post)

In [ ]:
# We will use the 'pre' version.
X_toy = padded_pre
y_toy = np.array(labels)

print("X_toy shape:", X_toy.shape, " <- (sentences, timesteps)")
print("y_toy shape:", y_toy.shape)

In [ ]:
# See the padding visually. Dark cells are padding zeros.
plt.imshow(X_toy, aspect="auto", cmap="Blues")
plt.colorbar(label="word index")
plt.xlabel("Timestep (word position)")
plt.ylabel("Sentence number")
plt.title("The padded data: every row is now 6 numbers long")
plt.grid(False)
plt.show()

`X_toy` is now **2D**: `(8, 6)`. Eight sentences, six timesteps each.

We are one step away from the 3D box the RNN wants.

---
# Part 8. Step 6 - The Embedding layer makes it 3D

We have `(8, 6)`. The RNN needs `(8, 6, features)`.
So each word number must become a **vector** of numbers.

There are two ways to do this.

### Way 1: One-hot encoding (simple, but wasteful)

Each word becomes a vector as long as the whole vocabulary, with a single 1 in it.
With a vocabulary of 30,000 words, every single word becomes 30,000 numbers.
That is far too big, and the vectors carry no meaning.

### Way 2: Embedding layer (what everyone actually uses)

Each word becomes a short vector, for example 8 or 100 numbers.
These numbers are **learned during training**. Words used in similar ways end up
with similar vectors. This is the same idea that later becomes Word2Vec, GloVe,
and the input of every transformer.

In [ ]:
# Way 1: one-hot. Look at how big it gets.
vocab_size = len(word_index)
one_hot = keras.utils.to_categorical(X_toy, num_classes=vocab_size)

print("Vocabulary size :", vocab_size)
print("One-hot shape   :", one_hot.shape, " <- (sentences, timesteps, vocab_size)")
print("Numbers stored  :", one_hot.size)

In [ ]:
# One word as a one-hot vector: mostly zeros.
print("Word index 3 as one-hot:")
print(one_hot[0][5].astype(int))

In [ ]:
# Way 2: the Embedding layer.
EMBED_DIM = 8   # each word becomes 8 numbers

embedding_layer = keras.layers.Embedding(input_dim=vocab_size, output_dim=EMBED_DIM)

embedded = embedding_layer(X_toy)

print("Input shape   :", X_toy.shape,          " <- 2D")
print("Output shape  :", embedded.shape, " <- 3D, ready for the RNN")

In [ ]:
# Compare the cost of the two ways.
print("One-hot numbers per word  :", vocab_size)
print("Embedding numbers per word:", EMBED_DIM)
print()
print("One-hot total numbers     :", one_hot.size)
print("Embedding total numbers   :", int(np.prod(embedded.shape)))

In [ ]:
# Look at one sentence after embedding: 6 words, 8 numbers each.
first_sentence = embedded[0].numpy()

print("Shape of one embedded sentence:", first_sentence.shape)
print()
print(np.round(first_sentence, 3))

In [ ]:
# The same thing as a picture.
plt.imshow(first_sentence, aspect="auto", cmap="RdBu")
plt.colorbar(label="value")
plt.xlabel("Embedding dimension (8 features)")
plt.ylabel("Timestep (6 words)")
plt.title("One sentence as the RNN sees it")
plt.grid(False)
plt.show()

These values look random right now, because the embedding has not been
trained yet. During training they move, and words with similar meaning drift
towards each other.

**We now have the 3D box: `(8, 6, 8)` = (sentences, timesteps, features).**

---
# Part 9. The whole journey in one picture

Run the cell below to see every step we just did, with the shape at each stage.

In [ ]:
steps = [
    "Raw text\n\"The movie was great!\"",
    "Cleaned\n\"the movie was great\"",
    "Tokens\n['the','movie','was','great']",
    "Numbers\n[2, 5, 3, 9]",
    "Padded (2D)\nshape (8, 6)",
    "Embedded (3D)\nshape (8, 6, 8)",
    "SimpleRNN\nreads 6 timesteps",
    "Output\n1 number: positive?",
]

fig, ax = plt.subplots(figsize=(8, 11))

for i in range(len(steps)):
    y = len(steps) - i
    ax.add_patch(plt.Rectangle((0.1, y - 0.35), 0.8, 0.7,
                               fill=True, facecolor="#eaf1fb",
                               edgecolor="#1c6fd4", linewidth=1.5))
    ax.text(0.5, y, steps[i], ha="center", va="center", fontsize=11)
    if i < len(steps) - 1:
        ax.annotate("", xy=(0.5, y - 0.38), xytext=(0.5, y - 0.65),
                    arrowprops=dict(arrowstyle="<-", color="#555"))

ax.set_xlim(0, 1)
ax.set_ylim(0.2, len(steps) + 0.7)
ax.axis("off")
ax.set_title("How a sentence reaches the RNN", fontsize=13, pad=16)
plt.show()

---
# Part 10. Step 7 - Feed it to a SimpleRNN

Now we build the smallest possible RNN and watch the shapes move through it.

A Keras model does the embedding for you. You do not pass the 3D array yourself:
you pass the 2D padded array, and the `Embedding` layer inside the model turns it
into 3D.

In [ ]:
RNN_UNITS = 4   # the size of the RNN memory

toy_model = keras.Sequential([
    keras.layers.Input(shape=(MAX_LEN,)),
    keras.layers.Embedding(input_dim=vocab_size, output_dim=EMBED_DIM),
    keras.layers.SimpleRNN(RNN_UNITS),
    keras.layers.Dense(1, activation="sigmoid"),
])

toy_model.summary()

### Reading the summary

| Layer | Output shape | What it means |
| --- | --- | --- |
| `Embedding` | `(None, 6, 8)` | 6 timesteps, 8 features per timestep. `None` means "any number of sentences". |
| `SimpleRNN` | `(None, 4)` | The whole sentence is squeezed into 4 numbers: the final memory. |
| `Dense` | `(None, 1)` | One number between 0 and 1: the probability of a positive review. |

### Where the parameters come from

* Embedding: `vocab_size x 8`
* SimpleRNN: `(8 x 4) + (4 x 4) + 4` = input weights + memory weights + bias
* Dense: `(4 x 1) + 1`

In [ ]:
# Check the SimpleRNN parameter count by hand.
input_weights  = EMBED_DIM * RNN_UNITS
memory_weights = RNN_UNITS * RNN_UNITS
biases         = RNN_UNITS

print("Input weights  :", input_weights)
print("Memory weights :", memory_weights)
print("Biases         :", biases)
print("Total          :", input_weights + memory_weights + biases)

In [ ]:
# return_sequences=False (the default): you get only the LAST memory.
rnn_last = keras.layers.SimpleRNN(RNN_UNITS, return_sequences=False)
out_last = rnn_last(embedded)
print("return_sequences=False ->", out_last.shape, " (one vector per sentence)")

In [ ]:
# return_sequences=True: you get the memory at EVERY timestep.
rnn_all = keras.layers.SimpleRNN(RNN_UNITS, return_sequences=True)
out_all = rnn_all(embedded)
print("return_sequences=True  ->", out_all.shape, " (one vector per word)")

**When to use which**

* `return_sequences=False` - one answer for the whole sequence.
  Sentiment, spam detection, topic classification.
* `return_sequences=True` - one answer per timestep, or when you want to stack
  another RNN layer on top. Text generation, named entity recognition.

If you stack two RNN layers, the first one **must** use `return_sequences=True`,
otherwise the second layer receives 2D input and Keras will raise an error.

In [ ]:
# Train the tiny model, just to prove the pipeline runs end to end.
toy_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

toy_history = toy_model.fit(X_toy, y_toy, epochs=30, verbose=0)

print("Final training accuracy:", round(toy_history.history["accuracy"][-1], 3))

In [ ]:
plt.plot(toy_history.history["loss"])
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("The toy model is learning (loss goes down)")
plt.show()

With 8 sentences the model simply memorises them. That is fine. The point
was the **pipeline**, not the accuracy. Now we do it with real data.

---
# Part 11. A real dataset: IMDB movie reviews

IMDB is a classic dataset of 50,000 movie reviews, split evenly into positive
and negative. It ships with Keras, so there is no file to download by hand.

One helpful detail: **Keras gives you this dataset already converted to numbers.**
Steps 1 to 4 (clean, tokenize, build vocabulary, encode) have already been done
for you. So here we start at Step 5, padding.

We will also decode some reviews back into words, so you can confirm the numbers
really are text.

In [ ]:
from keras.datasets import imdb

VOCAB_SIZE = 10000   # keep only the 10,000 most common words

(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = imdb.load_data(num_words=VOCAB_SIZE)

print("Training reviews:", len(x_train_raw))
print("Test reviews    :", len(x_test_raw))

In [ ]:
# One review is a list of numbers, not text.
print("First 20 numbers of review 0:")
print(x_train_raw[0][:20])
print()
print("Its label:", y_train_raw[0], " (1 = positive, 0 = negative)")

In [ ]:
# Reviews have very different lengths.
print("Review 0 length:", len(x_train_raw[0]))
print("Review 1 length:", len(x_train_raw[1]))
print("Review 2 length:", len(x_train_raw[2]))

### Decoding the numbers back into words

The IMDB index is shifted by 3, because three slots are reserved:
`0` = padding, `1` = start of sequence, `2` = unknown word.
So the real word for a number `n` is found at `n - 3` in the dictionary.

In [ ]:
imdb_word_index = imdb.get_word_index()

# Build the reverse dictionary: number -> word.
imdb_index_word = {}
for word in imdb_word_index:
    imdb_index_word[imdb_word_index[word] + 3] = word

imdb_index_word[0] = "<PAD>"
imdb_index_word[1] = "<START>"
imdb_index_word[2] = "<OOV>"

print("Dictionary size:", len(imdb_index_word))

In [ ]:
# Decode review 0 back into readable text.
decoded = []
for number in x_train_raw[0]:
    if number in imdb_index_word:
        decoded.append(imdb_index_word[number])
    else:
        decoded.append("<OOV>")

print(" ".join(decoded[:60]), "...")

In [ ]:
# Check the label balance of the training set.
print(pd.Series(y_train_raw).value_counts())

plt.bar(["negative (0)", "positive (1)"],
        [int((y_train_raw == 0).sum()), int((y_train_raw == 1).sum())],
        color=["#c0504d", "#4f81bd"])
plt.ylabel("Number of reviews")
plt.title("The classes are perfectly balanced")
plt.show()

In [ ]:
# How long are the reviews? This plot chooses our maxlen.
review_lengths = []
for review in x_train_raw:
    review_lengths.append(len(review))

review_lengths = np.array(review_lengths)

print("Shortest :", review_lengths.min())
print("Longest  :", review_lengths.max())
print("Average  :", int(review_lengths.mean()))
print("Median   :", int(np.median(review_lengths)))

In [ ]:
plt.hist(review_lengths, bins=60, range=(0, 1200), edgecolor="black")
plt.axvline(200, color="red", linestyle="--", label="maxlen = 200")
plt.xlabel("Words in a review")
plt.ylabel("Number of reviews")
plt.title("Review lengths in the IMDB training set")
plt.legend()
plt.show()

In [ ]:
# What fraction of reviews fit inside 200 words?
fits = (review_lengths <= 200).mean()
print("Reviews of 200 words or fewer:", round(fits * 100, 1), "%")

A longer `maxlen` keeps more text but makes training slower, because the RNN
has to take one step per timestep. 200 is a good balance for a class.

In [ ]:
# Step 5 again: padding. Exactly the same function as the toy example.
MAX_LEN_IMDB = 200

x_train_full = pad_sequences(x_train_raw, maxlen=MAX_LEN_IMDB,
                             padding="pre", truncating="post")
x_test = pad_sequences(x_test_raw, maxlen=MAX_LEN_IMDB,
                       padding="pre", truncating="post")

print("x_train_full shape:", x_train_full.shape)
print("x_test shape      :", x_test.shape)

In [ ]:
# Look at one padded review. Notice the zeros at the start.
print(x_train_full[1][:40])

In [ ]:
# Split the training data into train and validation.
from sklearn.model_selection import train_test_split

x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_raw,
    test_size=0.2, random_state=42, stratify=y_train_raw
)

print("Train      :", x_train.shape, y_train.shape)
print("Validation :", x_val.shape, y_val.shape)
print("Test       :", x_test.shape, y_test_raw.shape)

In [ ]:
# To keep training fast in class, use a smaller slice of the training data.
# Set USE_SUBSET = False if you have a GPU and want the full dataset.
USE_SUBSET = True
SUBSET_SIZE = 10000

if USE_SUBSET:
    x_train = x_train[:SUBSET_SIZE]
    y_train = y_train[:SUBSET_SIZE]

print("Training on:", x_train.shape[0], "reviews")

---
# Part 12. Build and train the RNN

The model is the same shape as the toy one. Only the numbers are bigger.

In [ ]:
EMBED_DIM_IMDB = 32
RNN_UNITS_IMDB = 32

model = keras.Sequential([
    keras.layers.Input(shape=(MAX_LEN_IMDB,)),
    keras.layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM_IMDB),
    keras.layers.SimpleRNN(RNN_UNITS_IMDB),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation="sigmoid"),
])

model.summary()

Check the shapes in the summary against the rule from Part 1:

* Input: `(None, 200)` - 200 timesteps
* After Embedding: `(None, 200, 32)` - **the 3D box**
* After SimpleRNN: `(None, 32)` - the final memory of the whole review
* After Dense: `(None, 1)` - the prediction

`Dropout` randomly switches off some values during training. It helps the model
avoid memorising the training data.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

print("Model compiled.")

In [ ]:
# Stop early if the validation loss stops improving.
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=2, restore_best_weights=True
)

history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=5,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1,
)

In [ ]:
# Plot accuracy and loss side by side.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="validation")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="validation")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_title("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

**How to read these plots**

* Both lines improving together: good, keep training.
* Training keeps improving while validation gets worse: overfitting. Stop, or add
  more dropout, or get more data.
* Neither line improves: the learning rate may be wrong, or the model too small.

---
# Part 13. Evaluate and predict your own sentence

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test_raw, verbose=0)

print("Test loss     :", round(test_loss, 4))
print("Test accuracy :", round(test_accuracy, 4))

In [ ]:
# Turn the probabilities into 0 / 1 predictions.
probabilities = model.predict(x_test, verbose=0)
predictions = (probabilities > 0.5).astype(int).flatten()

print("First 10 predictions :", predictions[:10])
print("First 10 true labels :", y_test_raw[:10])

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_test_raw, predictions)
print(cm)

In [ ]:
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks([0, 1], ["predicted negative", "predicted positive"])
plt.yticks([0, 1], ["true negative", "true positive"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i][j], ha="center", va="center", fontsize=14)
plt.title("Confusion matrix")
plt.grid(False)
plt.show()

In [ ]:
print(classification_report(y_test_raw, predictions,
                            target_names=["negative", "positive"]))

### Predicting your own review

This is the important cell. To predict on new text, you must repeat **exactly the
same steps** you used on the training data, in the same order. If you change one
step, the numbers mean something different and the prediction is nonsense.

In [ ]:
# Step 1 and 2: clean and split the new review.
my_review = "This film was absolutely wonderful and the acting was superb"

my_clean = my_review.lower()
cleaned = ""
for ch in my_clean:
    if ch not in string.punctuation:
        cleaned = cleaned + ch
my_tokens = cleaned.split()

print(my_tokens)

In [ ]:
# Step 3 and 4: use the IMDB vocabulary, with the +3 shift.
my_numbers = [1]   # 1 is the <START> token

for word in my_tokens:
    if word in imdb_word_index and imdb_word_index[word] + 3 < VOCAB_SIZE:
        my_numbers.append(imdb_word_index[word] + 3)
    else:
        my_numbers.append(2)   # 2 is <OOV>

print(my_numbers)

In [ ]:
# Step 5: pad to exactly the same length as training.
my_padded = pad_sequences([my_numbers], maxlen=MAX_LEN_IMDB,
                          padding="pre", truncating="post")

print("Shape going into the model:", my_padded.shape)

In [ ]:
# Step 6 and 7 happen inside the model.
score = model.predict(my_padded, verbose=0)[0][0]

print("Review :", my_review)
print("Score  :", round(float(score), 4))
if score > 0.5:
    print("Result : POSITIVE")
else:
    print("Result : NEGATIVE")

In [ ]:
# Try a few reviews at once. Change these and run again.
my_reviews = [
    "A brilliant movie with a beautiful story",
    "Terrible acting and a boring plot",
    "It was okay, nothing special",
]

for review in my_reviews:
    text = review.lower()
    cleaned = ""
    for ch in text:
        if ch not in string.punctuation:
            cleaned = cleaned + ch

    numbers = [1]
    for word in cleaned.split():
        if word in imdb_word_index and imdb_word_index[word] + 3 < VOCAB_SIZE:
            numbers.append(imdb_word_index[word] + 3)
        else:
            numbers.append(2)

    padded = pad_sequences([numbers], maxlen=MAX_LEN_IMDB,
                           padding="pre", truncating="post")
    s = model.predict(padded, verbose=0)[0][0]

    print(round(float(s), 3), " ", review)

---
# Part 14. SimpleRNN vs LSTM vs GRU

A `SimpleRNN` forgets. By the time it reaches word 200, it has almost lost word 1.
This is the vanishing gradient problem.

`LSTM` and `GRU` fix this with gates that decide what to keep and what to forget.
The good news for you: **the data pipeline does not change at all.** You swap one
layer and everything else stays the same.

In [ ]:
lstm_model = keras.Sequential([
    keras.layers.Input(shape=(MAX_LEN_IMDB,)),
    keras.layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM_IMDB),
    keras.layers.LSTM(RNN_UNITS_IMDB),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation="sigmoid"),
])

lstm_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
lstm_model.summary()

In [ ]:
lstm_history = lstm_model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=3,
    batch_size=128,
    verbose=1,
)

In [ ]:
lstm_loss, lstm_accuracy = lstm_model.evaluate(x_test, y_test_raw, verbose=0)

print("SimpleRNN test accuracy :", round(test_accuracy, 4))
print("LSTM test accuracy      :", round(lstm_accuracy, 4))

In [ ]:
plt.bar(["SimpleRNN", "LSTM"], [test_accuracy, lstm_accuracy],
        color=["#9e9e9e", "#4f81bd"])
plt.ylabel("Test accuracy")
plt.ylim(0, 1)
plt.title("LSTM usually beats SimpleRNN on long text")
plt.show()

Notice the parameter counts in the two summaries. An LSTM has about **four
times** the parameters of a SimpleRNN with the same number of units, because it
has four internal gates instead of one. A GRU sits in between, with three gates.

Try it yourself: copy the LSTM cell, change `keras.layers.LSTM` to
`keras.layers.GRU`, and compare all three.

---
# Part 15. Common mistakes

| Mistake | What you see | The fix |
| --- | --- | --- |
| Passing 2D data straight to an RNN without an Embedding layer | `expected ndim=3, found ndim=2` | Add `Embedding`, or `reshape` your numeric data to 3D |
| Stacking two RNN layers | `expected ndim=3, found ndim=2` | The first layer needs `return_sequences=True` |
| `maxlen` different at training and prediction | Bad predictions, no error | Save `maxlen` and reuse the same value everywhere |
| Building the vocabulary on train **and** test data | Accuracy looks too good | Build it on the training set only |
| A word index larger than `input_dim` of the Embedding | `indices[...] is not in [0, vocab_size)` | Set `input_dim` to at least `vocab_size` |
| Forgetting to clean new text the same way | Nonsense predictions | Put your cleaning steps in one place and reuse them |
| Using `padding='post'` with a SimpleRNN on long text | Weaker accuracy | Use `padding='pre'` so real words sit near the end |
| Very long `maxlen` | Training takes forever | Look at the length histogram and pick a sensible cut |

### The two questions to ask whenever an RNN gives a shape error

1. What shape is my data right now? Print `.shape`.
2. What shape does the layer want? Check `model.summary()`.

Almost every RNN bug is the gap between those two answers.

---
# Part 16. Exercises

Do these yourself. Copy the cells you need and change them.

**Easy**

1. Change `MAX_LEN_IMDB` from 200 to 100, then to 400. What happens to accuracy
   and to training time?
2. Change `EMBED_DIM_IMDB` from 32 to 8, then to 128. Note the parameter count.
3. Write five of your own reviews and predict them. Find one the model gets wrong,
   and explain why you think it failed.

**Medium**

4. Replace `SimpleRNN` with `GRU` and compare all three models in one bar chart.
5. Add a second RNN layer. Remember `return_sequences=True` on the first one.
6. Wrap the LSTM in `keras.layers.Bidirectional` and see what the parameter count
   and the accuracy do.
7. Set `USE_SUBSET = False` and train on the full dataset. How much does the extra
   data help?

**Harder**

8. Rebuild the whole pipeline from raw text using `keras.layers.TextVectorization`
   instead of the manual vocabulary, so the cleaning lives inside the model.
9. Use the toy dataset to plot the embedding vectors in 2D after training, and see
   whether positive and negative words separate.
10. Take a dataset in your own language, for example Roman Urdu product reviews,
    and run the same seven steps on it. The steps do not change.

**When you have finished**

Push the notebook to GitHub with a README that explains the seven steps in your
own words. That README is worth more in an interview than the model itself.

---

### The seven steps, one last time

1. Clean the text
2. Split into tokens
3. Build the vocabulary
4. Turn words into numbers
5. Pad to one fixed length
6. Embed, so each word becomes a vector - the data is now 3D
7. Feed `(batch, timesteps, features)` into the RNN

Every text model you will ever build, including transformers, starts with some
version of these steps.